In [7]:
import xarray as xr
import numpy as np

file = "D:/Repositories/too_much_big_data/" + "CHL_DATA/daily_chl_2021-12-21.nc"
chl = xr.open_dataset(file, group="geophysical_data")  # chlor_a
nav = xr.open_dataset(file, group="navigation_data")   # latitude, longitude

chlor = chl["chlor_a"].values
lat2d = nav["latitude"].values
lon2d = nav["longitude"].values


target_lat, target_lon = 33.3, 125.5  # 예: 제주도 남쪽

d = np.hypot(lat2d - target_lat, lon2d - target_lon)
iy, ix = np.unravel_index(np.nanargmin(d), d.shape)


val = chlor[iy, ix]
actual_lat = lat2d[iy, ix]
actual_lon = lon2d[iy, ix]
print(f"value={val:.4f} mg/m³ @ ({actual_lat:.4f}, {actual_lon:.4f})")

value=nan mg/m³ @ (33.2995, 125.5012)


In [8]:
chl["chlor_a"]

<xarray.DataArray 'chlor_a' (lat: 8000, lon: 10500)> Size: 336MB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], dtype=float32)
Dimensions without coordinates: lat, lon
Attributes:
    units:      mg m^-3
    long_name:  Chlorophyll-a concentration

In [9]:
import xarray as xr
import dask
import os
from dask.diagnostics import ProgressBar
import re
import pandas as pd
from pathlib import Path

dask.config.set(scheduler='threads')

In [ ]:
data_root = Path('D:\\Repositories\\too_much_big_data\\CHL_DATA') # 파일 경로 작성 
paths = sorted(data_root.glob("daily_chl_*.nc"))

date_re = re.compile(r"daily_chl_(\d{4}-\d{2}-\d{2})\.nc$")

In [11]:
def extract_time(f: Path) -> pd.Timestamp:
    match_ = date_re.search(f.name)
    if not match_:
        raise ValueError(f"날짜를 파일명에서 찾지못함 {f.name}")
    y, m, d = map(int, match_.group(1).split('-'))
    return pd.Timestamp(year=y, month=m, day=d)

def add_time(ds: xr.Dataset, t):
    """time 추가"""
    with xr.open_dataset(ds.encoding['source'], group="geophysical_data", mask_and_scale=True) as chl:
        chl = chl.expand_dims(time=[t])
        return xr.Dataset({'chlor_a': chl})

In [12]:
test_time = extract_time(paths[0])
with xr.open_dataset(paths[0], group="navigation_data", mask_and_scale=True) as nav0:
    lat = nav0['latitude']
    lon = nav0['longitude']

In [ ]:
chunks = {"lat": 1000, "lon": 1000}
dataset = []
err_file = []
for p in paths:
    t = extract_time(p)
    try:
        ds = xr.open_dataset(p, group="geophysical_data", chunks=chunks)
        ds_a = ds['chlor_a'].expand_dims(time=[t])
        dataset.append(ds_a)
    except:
        print(t)
        err_file.append(t)
print(dataset)

# for f in paths:
#     time_date = extract_time(f)
#     if time_date is None:
#         continue
#     try:
#         chl = xr.open_dataset(f, group="geophysical_data", chunks={"lat": 1000, "lon": 1000}) 
#         nav = xr.open_dataset(f, group="navigation_data", chunks={"lat": 1000, "lon": 1000})  
#         lat = nav['latitude']
#         lon = nav["longitude"] 
#         ds = xr.Dataset({"chlor_a": chl}).assign_coords(lat=lat, lon=lon)
#         ds = ds.expand_dims(time=[time_date])
#         dataset.append(ds)
#     except:
#         print(time_date)
#         err_file.append(time_date)
err_file

2021-11-27 00:00:00


In [14]:
chl_time = xr.concat(dataset, dim='time').sortby("time")
print(chl_time)
chl_time = chl_time.to_dataset(name='chlor_a').assign_coords(lat=lat,lon=lon)
chl_time
# ds_all = xr.concat(dataset, dim='time')

# ds_all["time"] = pd.to_datetime(ds_all["time"].values)
monthly = chl_time.resample(time='MS').mean()
encoding = {
    "chlor_a" : {
        'zlib' : True, "complevel":4, # 압축
        "dtype" : "float32",
        "chunksizes": (1, 1000, 1000)
    }
}

import dask
with dask.config.set(scheduler='threads'):
    monthly.to_netcdf('chlor_a_monthly_2021-2025.nc', encoding=encoding, compute=True)

<xarray.DataArray 'chlor_a' (time: 1385, lat: 8000, lon: 10500)> Size: 465GB
dask.array<getitem, shape=(1385, 8000, 10500), dtype=float32, chunksize=(1, 100, 100), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 11kB 2021-11-03 2021-11-04 ... 2025-09-17
Dimensions without coordinates: lat, lon
Attributes:
    units:      mg m^-3
    long_name:  Chlorophyll-a concentration


KeyboardInterrupt: 